# Read-only CPU training audit
Select accelerator None. This reads actual Drive histories and embedded best states, prints all recorded metrics, and performs no inference or data writes. There are three latest checkpoints, not 15 archived snapshots.


In [ ]:
CODE_REVISION = '9b3accad2bcf3a76a77936aaea30e91469acfd48'
import os, sys, json, subprocess, tempfile
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
REPO=Path(tempfile.mkdtemp(prefix='audit-eval-',dir='/content'))/'repo'
subprocess.run(['git','clone','--no-checkout','https://github.com/firozehrazavi/eeg-alignment-phase1.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',CODE_REVISION],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==CODE_REVISION
WORK=Path('/content/drive/MyDrive/eeg_results/patch_phase1')
PREP=WORK/'pilot_sub01_200concepts_v1'
TRAINING=WORK/'controlled_sub01_five_epochs_v1'
# CPU-only child, no writes to training/prepared artifacts.
args=[sys.executable,'-m','phase1.final_audit','--training',str(TRAINING),'--audit']
result=subprocess.run(args,cwd=REPO,env=dict(os.environ,CUDA_VISIBLE_DEVICES=''),check=True,capture_output=True,text=True)
report=json.loads(result.stdout)
print('Validation candidate count:',report['validation_candidates'])
print('Validation Top-1 chance (%):',report['chance_top1_percent'])
print('Verified epoch histories:',report['verified_epoch_records'])
print('Timing log verified:',report['timing_records_verified'])
print(report['checkpoint_retention'])
print('Arm | Epoch | Loss | Validation Top-1 (%) | Seconds | Peak allocated GiB')
for arm,record in report['arms'].items():
    for row in record['history']:
        peak=row.get('peak_gpu_allocated_bytes')
        print(arm,row['epoch'],row['loss'],row['source_val_top1'],row.get('seconds','not recorded'),
              peak/2**30 if peak is not None else 'not recorded',sep=' | ')
    print('BEST:',arm,record['best_epoch'],record['best_validation_top1'],'%',record['checkpoint'],'embedded best_state')
print('Validation only. No final-test metrics computed.')
